In [ ]:
!pip install pandas torch torch_geometric chess numpy scikit-learn tqdm

In [ ]:
# Import necessary libraries
import pandas as pd  # For data manipulation
import torch
from torch_geometric.data import Data, DataLoader
import torch.nn.functional as F
from torch_geometric.nn import NNConv, global_mean_pool
import chess  # For parsing FEN strings
import numpy as np
from sklearn.model_selection import train_test_split
from torch import nn
from tqdm import tqdm  # For progress bars
import re  # For regular expressions

In [ ]:
# Function to convert evaluation scores
def parse_evaluation(evaluation_str):
    """
    Converts evaluation scores from strings to numerical values.
    Handles mate in x moves notation.
    """
    # Check if the evaluation is in mate notation
    mate_pattern = re.match(r'M(-?\d+)', str(evaluation_str))
    if mate_pattern:
        mate_moves = int(mate_pattern.group(1))
        # Map mate in x moves to a score between 5000 and -5000
        # M1 maps to 5000 or -5000
        # Decrease linearly as the number of moves to mate increases
        # For positive mate moves (e.g., M1, M2), the score decreases from 5000
        # For negative mate moves (e.g., M-1, M-2), the score increases from -5000
        max_score = 5000
        # Calculate the score decrement per move
        decrement_per_move = 5000 / 100  # Adjust '100' to change how quickly the score decreases
        if mate_moves > 0:
            # Positive mate moves (e.g., M1 means white mates in 1)
            score = max_score - (mate_moves - 1) * decrement_per_move
            score = max(score, 0)  # Ensure score does not go below 0
        else:
            # Negative mate moves (e.g., M-1 means black mates in 1)
            mate_moves = abs(mate_moves)
            score = -max_score + (mate_moves - 1) * decrement_per_move
            score = min(score, 0)  # Ensure score does not go above 0
    else:
        # It's a numerical evaluation
        score = float(evaluation_str)
    return score

# Read the dataset from CSV file
# Assuming the CSV has two columns: 'fen' and 'evaluation'
data = pd.read_csv('/kaggle/input/stockfish-best-moves-compilation/dataset_eval.csv')

# Convert evaluation scores
data['Evaluation'] = data['Evaluation'].apply(parse_evaluation)

# Normalize the evaluation scores to be between -1 and 1
# Assuming evaluation scores are between -5000 and 5000
data['Evaluation'] = data['Evaluation'] / 5000.0

# The rest of the code remains the same...

# Function to convert FEN string to graph data
# def fen_to_graph(fen, evaluation):
#     board = chess.Board(fen)

#     # Mapping from squares to node indices
#     square_to_node = {}
#     node_features = []
#     edge_index = [[], []]  # Edge indices for source and target nodes
#     edge_attr = []  # Edge attributes

#     node_idx = 0  # Node index counter

#     # First pass: create nodes for all squares with pieces
#     for square in chess.SQUARES:
#         piece = board.piece_at(square)
#         if piece:
#             # Map square to node index
#             square_to_node[square] = node_idx
#             # Encode piece type and color
#             # Piece type: 1 (Pawn) to 6 (King)
#             # Color: True for White, False for Black
#             piece_type = piece.piece_type
#             color = piece.color  # True for white, False for black
#             # Encode piece as integer: positive for white, negative for black
#             feature = [piece_type * (1 if color else -1)]
#             node_features.append(feature)
#             node_idx += 1

#     # Second pass: create nodes for empty squares reachable by legal moves
#     # and build edges based on legal moves
#     for move in board.legal_moves:
#         from_square = move.from_square
#         to_square = move.to_square

#         # Only consider moves from squares that have been mapped to nodes
#         if from_square in square_to_node:
#             source_node_idx = square_to_node[from_square]

#             # If the target square has a piece, it should already have a node
#             if to_square in square_to_node:
#                 target_node_idx = square_to_node[to_square]
#             else:
#                 # For empty squares, create a node with feature [0]
#                 square_to_node[to_square] = node_idx
#                 node_features.append([0])  # Use 0 to represent empty squares
#                 target_node_idx = node_idx
#                 node_idx += 1

#             # Add edge from source to target node
#             edge_index[0].append(source_node_idx)
#             edge_index[1].append(target_node_idx)

#             # Extract edge attributes
#             is_capture = 1 if board.is_capture(move) else 0
#             is_promotion = 1 if move.promotion else 0
#             is_castling = 1 if board.is_castling(move) else 0
#             is_en_passant = 1 if board.is_en_passant(move) else 0

#             # Edge attributes: [is_capture, is_promotion, is_castling, is_en_passant]
#             edge_attr.append([is_capture, is_promotion, is_castling, is_en_passant])
            
#         else:
#             print("missed " + str(move))

#     # Convert to tensors
#     x = torch.tensor(node_features, dtype=torch.float)
#     edge_index = torch.tensor(edge_index, dtype=torch.long)
#     edge_attr = torch.tensor(edge_attr, dtype=torch.float)
#     y = torch.tensor([evaluation], dtype=torch.float)

#     # Create Data object
#     data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
#     return data

import torch
import chess
from torch_geometric.data import Data

def fen_to_graph(fen, evaluation):
    board = chess.Board(fen)

    # Nodes for all squares, node index from 0 to 63
    node_features = []
    edge_index = [[], []]
    edge_attr = []

    # Piece value mapping with normalization (adjust scaling_factor as needed)
    piece_value_dict = {
        chess.PAWN: 1,
        chess.KNIGHT: 3,
        chess.BISHOP: 3,
        chess.ROOK: 5,
        chess.QUEEN: 9,
        chess.KING: 100  # High value for the king
    }
    scaling_factor = 100  # Adjust as needed

    for square in chess.SQUARES:
        piece = board.piece_at(square)
        if piece:
            piece_type = piece.piece_type
            color = piece.color  # True for white, False for black
            # Get piece value and scale
            piece_value = piece_value_dict[piece_type] / scaling_factor
            # Encode piece value with sign for color
            piece_feature = piece_value * (1 if color else -1)
        else:
            piece_feature = 0.0  # Empty square

        # Positional encoding: get file and rank indices
        file_idx = chess.square_file(square)  # 0 (file 'a') to 7 (file 'h')
        rank_idx = chess.square_rank(square)  # 0 (rank '1') to 7 (rank '8')

        # Normalize coordinates to range [0, 1]
        x_coord = file_idx / 7.0
        y_coord = rank_idx / 7.0

        # Combine features: [piece_feature, x_coord, y_coord]
        feature = [piece_feature, x_coord, y_coord]
        node_features.append(feature)

    # Build edges based on legal moves
    for move in board.legal_moves:
        from_square = move.from_square
        to_square = move.to_square

        source_node_idx = from_square
        target_node_idx = to_square

        # Add edge from source to target node
        edge_index[0].append(source_node_idx)
        edge_index[1].append(target_node_idx)

        # Extract edge attributes
        is_capture = int(board.is_capture(move))
        is_promotion = int(move.promotion is not None)
        is_castling = int(board.is_castling(move))
        is_en_passant = int(board.is_en_passant(move))

        # Edge attributes
        edge_attr.append([is_capture, is_promotion, is_castling, is_en_passant])

    # Convert to tensors
    x = torch.tensor(node_features, dtype=torch.float)  # Shape: [64, 3]
    edge_index = torch.tensor(edge_index, dtype=torch.long)
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    y = torch.tensor([evaluation], dtype=torch.float)

    # Determine the active color (turn)
    turn = torch.tensor([1.0]) if board.turn == chess.WHITE else torch.tensor([0.0])

    # Create Data object
    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, turn=turn)
    return data





# Modify the graph_list to store graphs with FEN strings
graph_list = []

for index, row in tqdm(data.iterrows(), total=len(data)):
    fen = row['Position']
    evaluation = row['Evaluation']
    try:
        graph = fen_to_graph(fen, evaluation)
        graph.fen = fen  # Store the FEN string in the graph data
        graph_list.append(graph)
    except Exception as e:
        print(f"Error processing FEN at index {index}: {e}")


# Split data into training and testing sets
train_graphs, test_graphs = train_test_split(graph_list, test_size=0.2, random_state=42)

print(len(train_graphs), len(test_graphs))

# Create DataLoaders for batching
batch_size = 128
train_loader = DataLoader(train_graphs, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_graphs, batch_size=batch_size, shuffle=False)
print(len(train_loader), len(test_loader))

In [ ]:
import matplotlib.pyplot as plt

# Extract evaluation scores from training and testing datasets
train_evaluations = [graph.y.item() for graph in train_graphs]
test_evaluations = [graph.y.item() for graph in test_graphs]

# Plot histograms
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(train_evaluations, bins=50, color='blue', alpha=0.7)
plt.title('Training Set Evaluation Scores')
plt.xlabel('Normalized Evaluation Score')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
plt.hist(test_evaluations, bins=50, color='green', alpha=0.7)
plt.title('Testing Set Evaluation Scores')
plt.xlabel('Normalized Evaluation Score')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
import networkx as nx
from torch_geometric.utils import to_networkx

# Visualize 5 sample graphs from the training set
sample_graphs = train_graphs[:5]

for i, graph in enumerate(sample_graphs):
    # Convert the torch_geometric Data object to a NetworkX graph
    G = to_networkx(graph, node_attrs=['x'], edge_attrs=['edge_attr'])

    # Get node labels (piece types or square names)
    node_labels = {}
    for idx, data in G.nodes(data=True):
        # Map the node index to a square name (e.g., 'e4')
        square_name = chess.square_name(idx)
        piece_value = data['x'][0]
        if piece_value == 0:
            label = f'{square_name}\nEmpty'
        else:
            piece_type = abs(int(piece_value))
            color = 'W' if piece_value > 0 else 'B'
            piece_names = ['P', 'N', 'B', 'R', 'Q', 'K']
            if 1 <= piece_type <= 6:
                label = f"{square_name}\n{color}{piece_names[piece_type - 1]}"
            else:
                label = f"{square_name}\n{color}?{piece_type}"
        node_labels[idx] = label

    # Get edge labels (move types)
    edge_labels = {}
    for u, v, data in G.edges(data=True):
        attrs = data['edge_attr']
        is_capture = int(attrs[0])
        is_promotion = int(attrs[1])
        is_castling = int(attrs[2])
        is_en_passant = int(attrs[3])
        move_types = []
        if is_capture:
            move_types.append('C')  # Capture
        if is_promotion:
            move_types.append('P')  # Promotion
        if is_castling:
            move_types.append('K')  # Castling
        if is_en_passant:
            move_types.append('E')  # En Passant
        if not move_types:
            move_types.append('N')  # Normal move
        edge_label = ','.join(move_types)
        edge_labels[(u, v)] = edge_label

    plt.figure(figsize=(12, 10))
    pos = nx.spring_layout(G, seed=42, k=0.15)  # Adjust 'k' for better spacing
    nx.draw(G, pos, with_labels=False, node_size=500, font_size=8)
    nx.draw_networkx_labels(G, pos, labels=node_labels, font_size=6)
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=6)
    plt.title(f"Graph {i+1} - Evaluation: {graph.y.item():.2f}\nFEN: {graph.fen}")
    print(graph.fen)
    plt.axis('off')
    plt.show()

In [ ]:
from torch_geometric.nn import NNConv, global_mean_pool, BatchNorm
import torch.nn.functional as F
import torch.nn as nn

class GCNRegressor(nn.Module):
    def __init__(self):
        super(GCNRegressor, self).__init__()
        # Input dimension is now 3 (piece feature, x_coord, y_coord)
        in_channels = 3
        hidden_channels = 128
        edge_attr_dim = 4

        # Edge networks for NNConv layers
        self.edge_nn1 = nn.Sequential(
            nn.Linear(edge_attr_dim, 64),
            nn.ReLU(),
            nn.Linear(64, in_channels * hidden_channels)
        )
        self.conv1 = NNConv(in_channels, hidden_channels, self.edge_nn1, aggr='mean')
        self.bn1 = BatchNorm(hidden_channels)

        self.edge_nn2 = nn.Sequential(
            nn.Linear(edge_attr_dim, 64),
            nn.ReLU(),
            nn.Linear(64, hidden_channels * hidden_channels)
        )
        self.conv2 = NNConv(hidden_channels, hidden_channels, self.edge_nn2, aggr='mean')
        self.bn2 = BatchNorm(hidden_channels)

        self.edge_nn3 = nn.Sequential(
            nn.Linear(edge_attr_dim, 64),
            nn.ReLU(),
            nn.Linear(64, hidden_channels * hidden_channels)
        )
        self.conv3 = NNConv(hidden_channels, hidden_channels, self.edge_nn3, aggr='mean')
        self.bn3 = BatchNorm(hidden_channels)

        # Fully connected layers
        self.fc1 = nn.Linear(hidden_channels + 1, 64)  # +1 for the turn feature
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

    def forward(self, data):
        # Unpack data
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch

        # Apply graph convolution and activation with batch normalization
        x = F.relu(self.bn1(self.conv1(x, edge_index, edge_attr)))
        x = F.relu(self.bn2(self.conv2(x, edge_index, edge_attr)))
        x = F.relu(self.bn3(self.conv3(x, edge_index, edge_attr)))

        # Global mean pooling to get graph-level representation
        x = global_mean_pool(x, batch)

        # Include the turn information
        turn = data.turn.view(-1, 1)  # Shape: [batch_size, 1]
        x = torch.cat([x, turn], dim=1)  # Concatenate along the feature dimension

        # Fully connected layers with dropout
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x.view(-1)



# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = GCNRegressor()

# model = torch.nn.DataParallel(model)
model = model.to(device)

# Instantiate the optimizer and loss function
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-2)
criterion = nn.MSELoss()  # Mean Squared Error for regression

# Define the cosine annealing scheduler
from torch.optim.lr_scheduler import CosineAnnealingLR

num_epochs = 1000  # Total number of epochs

scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=0)

# Training function
def train():
    model.train()
    total_loss = 0
    for data in train_loader:
        optimizer.zero_grad()
        data = data.to(device)
        output = model(data)
        # Compute loss
        loss = criterion(output, data.y.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs  # Accumulate loss
    return total_loss / len(train_loader.dataset)

# Testing function
def test(loader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            output = model(data)
            # Compute loss
            loss = criterion(output, data.y.view(-1))
            total_loss += loss.item() * data.num_graphs
    return total_loss / len(loader.dataset)

In [ ]:
# Training loop with cosine annealing scheduler
best_loss = float('inf')
for epoch in range(1, num_epochs + 1):
    train_loss = train()
    test_loss = test(test_loader)
    print(f'Epoch {epoch}, Train Loss: {train_loss:.6f}, Test Loss: {test_loss:.6f}')

    # Step the scheduler
    scheduler.step()

    # Save the model if test loss decreases
    if test_loss < best_loss:
        best_loss = test_loss
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"New best model saved at epoch {epoch}")

    # Print current learning rate
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Current Learning Rate: {current_lr}")